# pytorch实现RNN底层

# 1. 依赖包

In [22]:
import torch
import torch.nn as nn
import torch.optim as optim

# 2. 模型

In [23]:
class RNN(nn.Module):
    #需要参数  单词个数  单词转换后的维度(词嵌入维度)  隐藏层维度 输出维度
    def __init__(self,vocab_size,embedding_dim,hidden_dim,output_dim):
        super(RNN,self).__init__()
        #词嵌入  [m,n] -> [m,n,vocab_size]
        self.emnedding = nn.Embedding(vocab_size,embedding_dim)
        self.hidden_dim = hidden_dim

        #RNN
        #self.rnn = nn.RNN(embedding_dim,hidden_dim,batch_first=True)
        self.i2h = nn.Linear(embedding_dim,hidden_dim)
        self.h2h = nn.Linear(hidden_dim,hidden_dim)

        #输出层
        self.fc = nn.Linear(hidden_dim,output_dim)
        self.tanh = nn.Tanh()

    def forward(self,text):
        vocab_size = text.size(0)
        #几句话
        len = text.size(1)
        embedded = self.emnedding(text)

        hidden = torch.zeros(vocab_size,self.hidden_dim)

        #output,hidden = self.rnn(embedded)

        #最后一个 压平输入全连接
        #return self.fc(hidden.squeeze(0))

        
        for t in range(len):
            xt = embedded[:,t,:]
            # tanh(W_ih * xt + W_hh * h_{t-1}
            hidden = self.tanh(self.i2h(xt)+self.h2h(hidden))
            return self.fc(hidden)

## 3. 数据准备

In [24]:
inputs = torch.LongTensor([
    [1, 2, 4, 0, 0],
    [4, 3, 2, 1, 0],
    [1, 5, 6, 7, 2],
    [0, 0, 1, 2, 3]
])
labels = torch.LongTensor([1, 0, 1, 0])

## 4. 模型训练
### 4.1 模型实例化

In [25]:
model = RNN(10,8,16,2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.01)

### 4.2 训练

In [26]:
for epoch in range(50):
    model.train()

    #梯度清空
    optimizer.zero_grad()
    #前向传播
    predictions = model(inputs)
    # 损失
    loss = criterion(predictions,labels)
    # 反向传播
    loss.backward()

    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/50], Loss: {loss.item():.4f}')

Epoch [10/50], Loss: 0.2173
Epoch [20/50], Loss: 0.0265
Epoch [30/50], Loss: 0.0048
Epoch [40/50], Loss: 0.0019
Epoch [50/50], Loss: 0.0011


## 5. 推理

In [27]:
test_inputs = torch.LongTensor([
    [2, 5, 2, 5, 0],  
    [9, 8, 9, 0, 0]   
])

model.eval()

with torch.no_grad():
    output = model(test_inputs)
    #寻找列的最大值
    values,indices = torch.max(output,dim=1)
    for i in range(len(test_inputs)):
        pre_idx = indices[i].item()
        confidence = torch.softmax(output[i],dim=0)[pre_idx].item()
        label = "正面 (Positive)" if pre_idx == 1 else "负面 (Negative)"
    
        print(f"输入序列 {i+1}: {test_inputs[i].tolist()}")
        print(f"预测类别: {label}")
        print(f"置信度: {confidence:.2%}")
        print("-" * 20)

输入序列 1: [2, 5, 2, 5, 0]
预测类别: 负面 (Negative)
置信度: 98.34%
--------------------
输入序列 2: [9, 8, 9, 0, 0]
预测类别: 负面 (Negative)
置信度: 99.96%
--------------------
